In [2]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [4]:
# from ta.trend import macd,cci,adx,macd_signal,adx_pos,adx_neg
# from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
# def calculate(pd: pd.DataFrame,predict=True):
#     pdrsi = rsi(pd['close'],14)
#     # rsi.dropna(axis=0,inplace=True)
#     pdcci = cci(pd['high'],pd['low'],pd['close'],14)
#     # cci.dropna(axis=0,inplace=True)
#     pdadx = adx(pd['high'],pd['low'],pd['close'])
#     pdadx_pos = adx_pos(pd['high'],pd['low'],pd['low']) 
#     pdadx_neg = adx_neg(pd['high'],pd['low'],pd['low'])
#     # adx.dropna(axis=0,inplace=True)
#     pdmacd = macd(pd['close'])
#     # macd.dropna(axis=0,inplace=True)
#     pdmacd_signal = macd_signal(pd['close'])
#     # macd_signal.dropna(axis=0,inplace=True)
#     pdstochrsi_d = stochrsi_d(pd['close'])
#     pdstochrsi_k = stochrsi_k(pd['close'])
#     pdstochrsi = stochrsi(pd['close'])
#     # stochrsi.dropna(axis=0,inplace=True)
#     pd2 = pd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
#     pd2['adx'] = pdadx
#     pd2['adx_pos'] = pdadx_pos
#     pd2['adx_neg'] = pdadx_neg
#     pd2['macd'] = pdmacd
#     pd2['macd_signal'] = pdmacd_signal
#     pd2['stochrsi_d'] = pdstochrsi_d
#     pd2['stochrsi_k'] = pdstochrsi_k
#     pd2['stochrsi'] = pdstochrsi
#     pd2['RSI'] = pdrsi.apply(lambda x: 1 if x < 30 else 2 if x > 70 else 0).astype(int)
#     pd2['CCI'] = pdcci.apply(lambda x: 1 if x < -100 else 2 if x > 100 else 0).astype(int)
#     conditions_3 = (pd2['stochrsi'] > 0.80) & (pd2['stochrsi_k'] < pd2['stochrsi_d'])
#     conditions_4 = (pd2['stochrsi'] < 0.20) & (pd2['stochrsi_k'] > pd2['stochrsi_d'])
#     pd2['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0)).astype(int)
#     pd2['ADX'] = np.where((pd2['adx'] > 25.00) & (pd2['adx_pos'] > pd2['adx_neg']), 1, np.where((pd2['adx'] > 25.00) & (pd2['adx_pos'] < pd2['adx_neg']), 2, 0)).astype(int)
#     pd2['MCAD'] = np.where((pd2['macd'] > pd2['macd_signal']), 1, np.where(pd2['macd'] < pd2['macd_signal'], 2, 0)).astype(int)
#     if predict:
#         pd2['next_close'] = pd2['close'].shift(-1)
#     pd2.dropna(axis=0,inplace=True)
#     pd2.drop(columns=['adx','adx_pos','adx_neg','macd','macd_signal','stochrsi','stochrsi_d','stochrsi_k'],axis=1,inplace=True)
#     if predict:
#         counts_1 = pd2[['RSI', 'CCI', 'STOCH.RSI', 'ADX', 'MCAD']].eq(1).sum(axis=1)
#         counts_2 = pd2[['RSI', 'CCI', 'STOCH.RSI', 'ADX', 'MCAD']].eq(2).sum(axis=1)
#         pd2['Prediction'] = np.where((counts_1 > 2) & (pd2['open'] < pd2['next_close']), 1,
#                               np.where((counts_2 > 2) & (pd2['open'] > pd2['next_close']), 2, 0))
#         # condition = [pd2['RSI'],pd2['CCI'],pd2['STOCH.RSI'],pd2['ADX'],pd2['MCAD']]
#         # pd2['Prediction'] = np.where((condition.count(1) > 2) & (pd2['open'] < pd2['next_close']), 1,
#         #                         np.where((condition.count(2) > 2) & (pd2['open'] > pd2['next_close']), 2, 0)).astype(int)
#         pd2.drop(columns=['next_close'], inplace=True)
#     return pd2


In [5]:
from ta.trend import macd,cci,adx,macd_signal,adx_pos,adx_neg,sma_indicator
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
def calculate_1(pd: pd.DataFrame,predict=True):
    pdrsi = rsi(pd['close'],14)
    # rsi.dropna(axis=0,inplace=True)
    pdcci = cci(pd['high'],pd['low'],pd['close'],14)
    # cci.dropna(axis=0,inplace=True)
    pdadx = adx(pd['high'],pd['low'],pd['close'])
    pdadx_pos = adx_pos(pd['high'],pd['low'],pd['low']) 
    pdadx_neg = adx_neg(pd['high'],pd['low'],pd['low'])
    # adx.dropna(axis=0,inplace=True)
    pdmacd = macd(pd['close'])
    # macd.dropna(axis=0,inplace=True)
    pdmacd_signal = macd_signal(pd['close'])
    # macd_signal.dropna(axis=0,inplace=True)
    pdstochrsi_d = stochrsi_d(pd['close'])
    pdstochrsi_k = stochrsi_k(pd['close'])
    pdstochrsi = stochrsi(pd['close'])
    # stochrsi.dropna(axis=0,inplace=True)
    pd2 = pd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
    pd2['rsi'] = pdrsi
    pd2['cci'] = pdcci
    pd2['adx'] = pdadx
    pd2['adx_pos'] = pdadx_pos
    pd2['adx_neg'] = pdadx_neg
    pd2['macd'] = pdmacd
    pd2['macd_signal'] = pdmacd_signal
    pd2['stochrsi_d'] = pdstochrsi_d
    pd2['stochrsi_k'] = pdstochrsi_k
    pd2['stochrsi'] = pdstochrsi
    # pd2['RSI'] = pdrsi.apply(lambda x: 1 if x < 30 else 2 if x > 70 else 0).astype(int)
    # pd2['CCI'] = pdcci.apply(lambda x: 1 if x < -100 else 2 if x > 100 else 0).astype(int)
    # conditions_3 = (pd2['stochrsi'] > 80) & (pd2['stochrsi_k'] < pd2['stochrsi_d'])
    # conditions_4 = (pd2['stochrsi'] < 20) & (pd2['stochrsi_k'] > pd2['stochrsi_d'])
    # pd2['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0)).astype(int)
    pd2['ADX'] = np.where((pd2['adx'] > 25.00) & (pd2['adx_pos'] > pd2['adx_neg']), 1, np.where((pd2['adx'] > 25.00) & (pd2['adx_pos'] < pd2['adx_neg']), 2, 0)).astype(int)
    # pd2['MCAD'] = np.where((pd2['macd'] > pd2['macd_signal']), 1, np.where(pd2['macd'] < pd2['macd_signal'], 2, 0)).astype(int)
    pd2['next_close'] = pd2['close'].shift(-1)
    pd2.dropna(axis=0,inplace=True)
    # pd2.drop(columns=['adx','adx_pos','adx_neg','macd','macd_signal','stochrsi','stochrsi_d','stochrsi_k'],axis=1,inplace=True)
    # condition = (sum([pd2['RSI'],pd2['CCI'],pd2['STOCH.RSI'],pd2['ADX'],pd2['MCAD']]) > 2)
    pd2['Prediction'] = np.where((pd2['open'] < pd2['next_close']), 1,
                            np.where((pd2['open'] > pd2['next_close']), 2, 0)).astype(int)
    pd2.drop(columns=['next_close','ADX'], inplace=True)
    return pd2

In [6]:
pd_data = list()
pd_data_1 = list()
first_list = ['EURUSD','EURCAD','EURJPY','EURGBP','EURAUD']
for curr in first_list:
    d = pd.read_csv(f"common/MachineLearningModel/output/{curr}_5_Min.csv")
    # cal = calculate(d)
    # pd_data.append(cal)d)
    # pd_data.append(cal)
    cal_1 = calculate_1(d)
    pd_data.append(cal_1)
sc_list = ['EURUSD','EURCAD','EURJPY','EURGBP','USDCAD','USDJPY']
for curr in sc_list:
    dd = pd.read_csv(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_1.csv')
    # cal = calculate(d)
    # pd_data.append(cal)
    cal_1 = calculate_1(d)
    pd_data.append(cal_1)
data = pd.concat(pd_data)
th_list = ['EURAUD','EURUSD','EURCAD','EURJPY','EURGBP','USDCAD','USDJPY']
for curr in sc_list:
    dd = pd.read_csv(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_2.csv')
    # cal = calculate(d)
    # pd_data.append(cal)
    cal_1 = calculate_1(d)
    pd_data.append(cal_1)
data = pd.concat(pd_data)
# data_1 = pd.concat(pd_data_1)

In [7]:
# print(data.iloc[:,6:].head())

In [8]:
# data['rsi'] = data['rsi'].astype(dtype=int)
# data['cci'] = data['cci'].astype(dtype=int)
# data['adx'] = data['adx'].astype(dtype=int)
# data['adx_pos'] = data['adx_pos'].astype(dtype=int)
# data['adx_neg'] = data['adx_neg'].astype(dtype=int)

In [9]:
# data.drop(axis=1,labels=['rsi','cci','adx','macd','macd_signal','stochrsi','stochrsi_k','stochrsi_d'],inplace=True)
# data.drop(axis=1,labels=['RSI_1'],inplace=True)

In [10]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.iloc[:,6:].head())
print(data.shape)


         rsi         cci        adx    adx_pos    adx_neg      macd  \
0  72.618496  150.645342  41.265595  15.666606   3.877951  0.000156   
1  72.919624  113.218606  41.923274  14.977224   4.929622  0.000165   
2  57.847187   63.500440  40.854860  13.680624   7.869523  0.000152   
3  52.531928    4.601077  38.889838  12.880078   9.847223  0.000132   
4  46.139992  -66.246057  36.421900  12.086663  13.183043  0.000104   

   macd_signal  stochrsi_d  stochrsi_k  stochrsi  Prediction  
0     0.000112    0.935348    0.935349  0.806048           2  
1     0.000123    0.932423    0.874758  0.818225           2  
2     0.000128    0.807033    0.610991  0.208699           2  
3     0.000129    0.609352    0.342308  0.000000           2  
4     0.000124    0.340955    0.069566  0.000000           2  
(107851, 17)


In [11]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


[0 1 2]


In [12]:
# import pickle
# combine_final_model = pickle.dump(stacking_clf, open('combineclassifier.sav','wb'))

In [13]:
print(data['Prediction'].value_counts())
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.columns)
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.50, random_state = 24, shuffle=False)



Prediction
2    53150
1    51971
0     2730
Name: count, dtype: int64
Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'adx_pos', 'adx_neg', 'macd', 'macd_signal', 'stochrsi_d', 'stochrsi_k',
       'stochrsi', 'Prediction'],
      dtype='object')
Index(['rsi', 'cci', 'adx', 'adx_pos', 'adx_neg', 'macd', 'macd_signal',
       'stochrsi_d', 'stochrsi_k', 'stochrsi'],
      dtype='object')
rsi            107851
cci            107851
adx            107851
adx_pos        107851
adx_neg        107851
macd           107851
macd_signal    107851
stochrsi_d     107851
stochrsi_k     107851
stochrsi       107851
dtype: int64


In [14]:
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.ensemble import StackingClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.datasets import make_classification
# # Define the base learners
# base_learners = [
#     ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
#     ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
#     ('xgb', XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2))
# ]
# # Define the meta-learner
# meta_learner = LogisticRegression()
# # Build the Stacking classifier
# stacking_clf = StackingClassifier(estimators=base_learners, final_estimator=meta_learner)
# # Train the Stacking classifier
# stacking_clf.fit(X_train, y_train)
# # Evaluate the model
# stacking_clf.score(X_test, y_test)

In [15]:
# final_comb_model = StackingClassifier(estimators=base_learners, final_estimator=meta_learner)
# final_comb_model.fit(X,y)

In [16]:
# print(data_1['Prediction'].value_counts())
# X = data_1.iloc[:,6:-1]
# y = data_1.iloc[:, -1]
# print(data_1.columns)
# print(X.columns)
# print(X.count())
# # print(y.head())
# X_train, X_test, y_train, y_test =train_test_split(
#   X, y, test_size = 0.30, random_state = 24, shuffle=False)

In [17]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by XGBoost Classifier: 97.91933240611961
Accuracy on test data by XGBoost Classifier: 99.98516485554279


In [18]:

X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.50, random_state = 24, shuffle=False)
rf_model = RandomForestClassifier(random_state=24)
# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on the test set
preds = rf_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by RandomForest Classifier\
: {accuracy_score(y_train, rf_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by RandomForest Classifier\
: {accuracy_score(y_test, preds)*100}")

Accuracy on train data by RandomForest Classifier: 100.0
Accuracy on test data by RandomForest Classifier: 100.0


In [19]:
final_rf_model = RandomForestClassifier(random_state=24)
final_rf_model.fit(X, y)

RandomForestClassifier(random_state=24)

In [20]:
final_xgb_model = XGBClassifier()
final_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [21]:
# import pickle
# xgb_final_model = pickle.dump(final_xgb_model, open('xgbclassifier_5.sav','wb'))
# rf_final_model = pickle.dump(final_rf_model, open('rfclassifier_5.sav','wb'))

In [22]:
from TradingDataGenerate import main
s = main.TvDatafeed('mageshragav1@gmail.com','Magesh1@')


error while signin
you are using nologin method, data you access may be limited


In [23]:
# def result(data):
#     pd_data_dump = calculate(data)
#     pd_data_dump.dropna(inplace=True)
#     pd_data_dump.reset_index()
#     print(pd_data_dump.iloc[-1])
#     data_1 = pd_data_dump.iloc[-1,5:-1].to_dict()
#     data_1 = pd.DataFrame({key: [int(value)] for key, value in data_1.items()})
#     print(data_1)
#     output_1 = final_comb_model.predict(pd.DataFrame(data_1))
#     print(output_1)

In [24]:
def result_1(data):
    pd_data_dump = calculate_1(data)
    pd_data_dump.dropna(inplace=True)
    pd_data_dump.reset_index()
    # print(pd_data_dump.iloc[-1])
    data_1 = pd_data_dump.iloc[-1,5:-1].to_dict()
    data_1 = pd.DataFrame({key: [value] for key, value in data_1.items()})
    print(pd_data_dump.iloc[-1,0:4])
    print(data_1)
    print('\n')
    output = final_xgb_model.predict(pd.DataFrame(data_1))
    print(f"xgb classifier prediction {output}")
    output = final_rf_model.predict(pd.DataFrame(data_1))
    print(f"random forest classifier prediction {output}")

In [25]:
# import random
# # symbols = random.choice(['EURUSD','EURJPY','GBPUSD','EURGBP'])
# EURUSD_data = s.get_hist(symbol="EURUSD",exchange='FX',interval=main.Interval.in_1_minute,n_bars=150,extended_session=False)
# EURUSD_data_1 = EURUSD_data.copy(deep=True)
# print('EURUSD')
# result(EURUSD_data)

In [35]:
EURUSD_data_1 = s.get_hist(symbol="EURUSD",exchange='FX',interval=main.Interval.in_5_minute,n_bars=150,extended_session=False)
result_1(EURUSD_data_1)
cci_os = cci(EURUSD_data_1['high'],EURUSD_data_1['low'],EURUSD_data_1['close'],21)
sma_trend_high = sma_indicator(EURUSD_data_1['high'],21)
sma_trend_low = sma_indicator(EURUSD_data_1['low'],21)
print(cci_os.iloc[-1], EURUSD_data_1.iloc[-1]['open'], sma_trend_high.iloc[-1])
if cci_os.iloc[-1] > 50 and sma_trend_high.iloc[-1] < EURUSD_data_1.iloc[-1]['open']:
    print('buy')
elif cci_os.iloc[-1] < -50 and sma_trend_low.iloc[-1] > EURUSD_data_1.iloc[-1]['open']:
    print('sell')
else:
    print('netrual')

open     1.06311
high     1.06369
low      1.06311
close    1.06355
Name: 2024-04-17 10:55:00, dtype: float64
         rsi         cci        adx    adx_pos    adx_neg      macd  \
0  44.144959 -107.592869  17.223899  11.429076  22.159698 -0.000157   

   macd_signal  stochrsi_d  stochrsi_k  stochrsi  
0    -0.000053    0.083385    0.146095  0.332967  


xgb classifier prediction [1]
random forest classifier prediction [1]
-85.61882626379534 1.06355 1.0643419047619047
sell


In [27]:
from tradingview_ta import TA_Handler, Interval, Exchange
import tradingview_ta
handler = TA_Handler(
    symbol="EURUSD",
    exchange="FX_IDC",
    screener="forex",
    interval="5m",
    timeout=None
)
ss = handler.get_analysis()
print(ss.oscillators)

{'RECOMMENDATION': 'NEUTRAL', 'BUY': 2, 'SELL': 1, 'NEUTRAL': 8, 'COMPUTE': {'RSI': 'NEUTRAL', 'STOCH.K': 'NEUTRAL', 'CCI': 'BUY', 'ADX': 'NEUTRAL', 'AO': 'NEUTRAL', 'Mom': 'BUY', 'MACD': 'SELL', 'Stoch.RSI': 'NEUTRAL', 'W%R': 'NEUTRAL', 'BBP': 'NEUTRAL', 'UO': 'NEUTRAL'}}
